In [1]:
import hail as hl
#import argparse
#import pandas
import os

from ukb_utils import hail_init
from ukb_utils import genotypes
from ko_utils import qc
from ko_utils import analysis

In [2]:
import hail as hl
from gnomad.utils.vep import process_consequences

In [3]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
hail_init.hail_bmrc_init('/logs/hail/hail_export.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compe047.hpc.in.bmrc.ox.ac.uk:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.74-0c3a74d12093
LOGGING: writing to /logs/hail/hail_export.log


In [12]:
mt = qc.get_table(genotypes.get_ukb_wes_200k_post_qc_path(chrom=22), 'mt')
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'rsid': str
    'qual': float64
    'filters': set<str>
    'info': struct {
        AF: float64, 
        AQ: array<int32>, 
        AC: int32, 
        AN: int32, 
        ExcessHet: float64
    }
----------------------------------------
Entry fields:
    'GT': call
    'DP': int32
    'AD': array<int32>
    'GQ': int32
    'PL': array<int32>
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [ ]:
# get non singletons (note: individuals are on 12788)
mt1 = qc.get_table('data/phased/ukb_wes_200k_phased_chr22.1of1.vcf.gz', 'vcf')
#mt1 = qc.get_table('data/tmp/ukb_wes_200k_phased_tmp_chr22.1of1.vcf.gz', 'vcf')
#mt1 = qc.recalc_info(mt1)
mt1 = qc.translate_sample_ids(mt1, from_app='12788', to_app='11867') # translate to lindgren app ID
#mt1 = qc.filter_to_european(mt1)
mt1 = qc.filter_min_missing(mt1, 0.05)
#mt1 = qc.filter_max_maf(mt1, float(0.02))
mt1 = analysis.annotate_vep(mt1, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
#mt1 = analysis.filter_vep(mt1, 'consequence_category',['damaging_missense','ptv'])
#mt1 = mt1.filter_rows(hl.agg.any(mt1.GT.is_non_ref()))

In [ ]:
def annotate_phased_entries(mt):
    r'''Annotates alleles that have the alternate allele on either first or second strand.'''
    mt = mt.annotate_entries(a0_alt = mt.GT ==  hl.parse_call('1|0'))
    mt = mt.annotate_entries(a1_alt = mt.GT ==  hl.parse_call('0|1'))
    mt = mt.annotate_entries(a_homo = mt.GT ==  hl.parse_call('1|1'))
    return mt


In [ ]:
mt1 = annotate_phased_entries(mt1)

In [ ]:
def annotate_compound_hetz(mt):
    
    mt = mt.filter_entries(~mt.GT.is_hom_var())
    ht0 = (mt.group_rows_by(mt.vep.Gene)
           .aggregate(s0 = hl.agg.any(mt.a0_alt))).entries()
    ht1 = (mt.group_rows_by(mt.vep.Gene)
           .aggregate(s1 = hl.agg.any(mt.a1_alt))).entries()
    ht = ht0.annotate(s1 = ht1[ht0.Gene, ht0.s].s1)
    ht = ht.filter((ht.s0 == True) & (ht.s1 == True))
    
    return None
    


In [ ]:
mt = mt1
compound_heterozygous = mt.a0_alt & mt.a0_alt
homozygous = mt.a_homo
mt = mt.filter_entries(~mt.GT.is_hom_var())

ht0 = (mt.group_rows_by(mt.vep.Gene)
        .aggregate(s0 = hl.agg.any(mt.a0_alt))).entries()
ht1 = (mt.group_rows_by(mt.vep.Gene)
        .aggregate(s1 = hl.agg.any(mt.a1_alt))).entries()

In [ ]:
ht = ht0.annotate(s1 = ht1[ht0.Gene, ht0.s].s1)
#ht.describe()
ht = ht.filter((ht.s0 == True) & (ht.s1 == True))
ht.show()

In [ ]:
mt2 = qc.get_table('data/unphased/unfiltered/ukb_wes_200k_filtered_chr22.mt','mt')
#mt2 = qc.get_table('data/unphased/tmp/ukb_wes_200k_filtered_2samples_chr22.vcf.bgz', 'vcf')
mt2 = qc.filter_to_european(mt2)
mt2 = qc.filter_max_mac(mt2, 1) # get singletons
mt2 = qc.filter_min_missing(mt2, 0.05)
mt2 = analysis.annotate_vep(mt2, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
mt2 = analysis.filter_vep(mt2, 'consequence_category',['damaging_missense','ptv'])


In [ ]:
out = analysis.construct_summary_mt(mt1)

In [ ]:
out.aggregate_entries(hl.agg.counter(out.ko))

In [ ]:
# Count burden per gene per individual
mt1_cat = analysis.gene_burden_category_annotations_per_sample(mt1)
mt2_cat = analysis.gene_burden_category_annotations_per_sample(mt2)

# combine singleton table and full table
res = mt1_cat.annotate_entries(singletons = mt2_cat[(mt1_cat.Gene, mt1_cat.consequence_category), mt1_cat.s].n)
res = res.annotate_entries(singletons = hl.if_else(hl.is_missing(res.singletons),0,res.singletons))
res = res.annotate_entries(total = res.n + res.singletons)
res = res.entries()
res = res.filter(res.total > 0)

In [ ]:
n = res.singletons.collect()

In [ ]:
n

In [ ]:
# why are there no singletons left in chr 22 after filtering?

# How many singletons / individuals at missingness 5%
mmt2 = mt2.filter_rows(hl.agg.any(mt2.GT.is_non_ref()))
#mmt1 = mt1.filter_rows(hl.agg.any(mt1.GT.is_non_ref()))

# 

In [ ]:
mmt2.count()

In [ ]:
# Count burden per gene per individual
mt1_cat = analysis.gene_burden_category_annotations_per_sample(mt1)
mt2_cat = analysis.gene_burden_category_annotations_per_sample(mt2)

# combine singleton table and full table
res = mt1_cat.annotate_entries(singletons = mt2_cat[(mt1_cat.Gene, mt1_cat.consequence_category), mt1_cat.s].n)
res = res.annotate_entries(singletons = hl.if_else(hl.is_missing(res.singletons),0,res.singletons))
res = res.annotate_entries(total = res.n + hl.if_else(hl.is_missing(res.singletons),0,res.singletons))
res = res.entries()
res = res.filter(res.singletons > 0)

In [ ]:
mt = analysis.get_prob_ko_matrix(mt1, mt2, field_drop = None)

In [ ]:
mt = qc.get_table('data/mt/ukb_wes_vep_200_chr22.mt','mt')
mt = analysis.annotate_vep(mt, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')

In [ ]:
mt.vep.decribe()

In [34]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
dataset = qc.get_table('data/unphased/tmp/ukb_wes_200k_filtered_2samples_chr22.vcf.bgz', 'vcf')
dataset = hl.vep(dataset, "/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/utils/configs/vep_env.json") 
dataset.vep.show()

2021-09-21 11:58:05 Hail: INFO: Coerced almost-sorted dataset
2021-09-21 11:59:48 Hail: INFO: Coerced almost-sorted dataset


FatalError: IOException: error=2, No such file or directory

Java stack trace:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 28.0 failed 1 times, most recent failure: Lost task 0.0 in stage 28.0 (TID 721) (compe093.hpc.in.bmrc.ox.ac.uk executor driver): java.io.IOException: Cannot run program "vep": error=2, No such file or directory
	at java.lang.ProcessBuilder.start(ProcessBuilder.java:1048)
	at is.hail.utils.richUtils.RichIterator$.pipe$extension(RichIterator.scala:87)
	at is.hail.methods.VEP.$anonfun$execute$5(VEP.scala:175)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:484)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:490)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at is.hail.utils.richUtils.RichContextRDD$$anon$1.hasNext(RichContextRDD.scala:71)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at scala

KeyboardInterrupt: 

In [ ]:
dataset.vep.worst_csq_by_gene_canonical.lof.show()

In [8]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
dataset = qc.get_table('data/unphased/tmp/ukb_wes_200k_filtered_2samples_chr22.vcf.bgz', 'vcf')
dataset = hl.vep(dataset, "/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/utils/configs/vep_env.json") 
dataset = process_consequences(dataset)
dataset.vep.worst_csq_by_gene_canonical.lof[0]

2021-09-21 12:30:13 Hail: INFO: Coerced sorted dataset


<StringExpression of type str>

In [30]:
# annotate with VEP
#hail_init.hail_bmrc_init('/logs/hail/hail_vep_export.log', 'GRCh38')
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/')
#dataset = qc.get_table('data/unphased/unfiltered/ukb_wes_200k_filtered_chr2.mt', 'mt')
dataset = qc.get_table('data/phased/ukb_wes_200k_phased_chr22.1of1.vcf.gz', 'vcf')

# clean up snpID and rsID
dataset = dataset.annotate_rows(rsid = dataset.rsid)
dataset = dataset.annotate_rows(snpid = dataset.rsid)
#dataset = qc.annotate_snpid(dataset)
#dataset = qc.annotate_rsid(dataset)
#dataset = qc.default_to_snpid_when_missing_rsid(dataset)

# Translate to lindgren IDs
#if True:
#    dataset = qc.translate_sample_ids(dataset, 12788, 11867)    
    
# recalc info
dataset = qc.recalc_info(dataset)

# get VEP
result = hl.vep(dataset, "/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb/utils/configs/vep_env.json") 
#result = process_consequences(result)

#qc.export_table(result, out_prefix = 'testtabledeleteme', out_type = 'mt')
result.vep.show()


FatalError: IOException: error=2, No such file or directory

Java stack trace:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 14.0 failed 1 times, most recent failure: Lost task 0.0 in stage 14.0 (TID 14) (compe093.hpc.in.bmrc.ox.ac.uk executor driver): java.io.IOException: Cannot run program "vep": error=2, No such file or directory
	at java.lang.ProcessBuilder.start(ProcessBuilder.java:1048)
	at is.hail.utils.richUtils.RichIterator$.pipe$extension(RichIterator.scala:87)
	at is.hail.methods.VEP.$anonfun$execute$5(VEP.scala:175)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:484)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:490)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at is.hail.utils.richUtils.RichContextRDD$$anon$1.hasNext(RichContextRDD.scala:71)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at scala.

FatalError: IOException: error=2, No such file or directory

Java stack trace:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 15.0 failed 1 times, most recent failure: Lost task 0.0 in stage 15.0 (TID 15) (compe093.hpc.in.bmrc.ox.ac.uk executor driver): java.io.IOException: Cannot run program "vep": error=2, No such file or directory
	at java.lang.ProcessBuilder.start(ProcessBuilder.java:1048)
	at is.hail.utils.richUtils.RichIterator$.pipe$extension(RichIterator.scala:87)
	at is.hail.methods.VEP.$anonfun$execute$5(VEP.scala:175)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:484)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:490)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at is.hail.utils.richUtils.RichContextRDD$$anon$1.hasNext(RichContextRDD.scala:71)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at scala.

In [31]:
#result.vep.describe()
result.vep.show()

FatalError: IOException: error=2, No such file or directory

Java stack trace:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 16.0 failed 1 times, most recent failure: Lost task 0.0 in stage 16.0 (TID 16) (compe093.hpc.in.bmrc.ox.ac.uk executor driver): java.io.IOException: Cannot run program "vep": error=2, No such file or directory
	at java.lang.ProcessBuilder.start(ProcessBuilder.java:1048)
	at is.hail.utils.richUtils.RichIterator$.pipe$extension(RichIterator.scala:87)
	at is.hail.methods.VEP.$anonfun$execute$5(VEP.scala:175)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:484)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:490)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at is.hail.utils.richUtils.RichContextRDD$$anon$1.hasNext(RichContextRDD.scala:71)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at scala.

FatalError: IOException: error=2, No such file or directory

Java stack trace:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 17.0 failed 1 times, most recent failure: Lost task 0.0 in stage 17.0 (TID 17) (compe093.hpc.in.bmrc.ox.ac.uk executor driver): java.io.IOException: Cannot run program "vep": error=2, No such file or directory
	at java.lang.ProcessBuilder.start(ProcessBuilder.java:1048)
	at is.hail.utils.richUtils.RichIterator$.pipe$extension(RichIterator.scala:87)
	at is.hail.methods.VEP.$anonfun$execute$5(VEP.scala:175)
	at scala.collection.Iterator$$anon$11.nextCur(Iterator.scala:484)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:490)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at is.hail.utils.richUtils.RichContextRDD$$anon$1.hasNext(RichContextRDD.scala:71)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at scala.collection.Iterator$$anon$11.hasNext(Iterator.scala:488)
	at scala.

In [13]:
result = analysis.variant_csqs_category_builder(result)
in_mt = in_mt.annotate_entries(rsid_entry = in_mt.rsid)
in_mt = in_mt.annotate_entries(snpid_entry = in_mt.snpid)
in_mt = in_mt.annotate_entries(rsid_gt = hl.delimit([in_mt.snpid_entry,
                                                         in_mt.rsid_entry,
                                                         hl.str(in_mt.GT),
                                                         in_mt.vep.consequence_category,
                                                         in_mt.vep.most_severe_consequence,
                                                         in_mt.vep.worst_csq_by_gene_canonical.lof,
                                                         in_mt.vep.worst_csq_by_gene_canonical.lof_flags,
                                                         hl.str(in_mt.dbnsfp.revel_score),
                                                         hl.str(in_mt.dbnsfp.cadd_phred_score)], ';'))

AttributeError: MatrixTable instance has no field, method, or property 'dbnsfp'
    Hint: use 'describe()' to show the names of all data fields.

In [ ]:
def filter_european(mt, only_annotate = False):
    r'''Get white british (app 11867) /well/lindgren/UKBIOBANK/DATA/QC/ukb_sqc_v2.txt
    and genetically european from /well/lindgren/UKBIOBANK/laura/k_means_clustering_pcs/ukbb_genetically_european_k4_4PCs_self_rep_Nov2020.txt''' 
    ht = hl.import_table('/well/lindgren/UKBIOBANK/laura/k_means_clustering_pcs/ukbb_genetically_european_k4_4PCs_self_rep_Nov2020.txt',
        types={'eid': hl.tstr, 'genetically_european': hl.tint32}).key_by('eid')
    mt = mt.annotate_cols(eur = ht[mt.s].genetically_european)
    undefined_eur = mt.aggregate_cols(hl.agg.sum(hl.is_missing(mt.eur)))
    pre_filter_count = mt.count()
    if undefined_eur == pre_filter_count[1]:
        raise ValueError('[get_genetically_european]: IDs for genetically europeans does not match keys in MatrixTable!')
    if undefined_eur > 0:
        print(f'[get_genetically_european]: Not all samples IDs mapped perfectly ({undefined_eur}/{pre_filter_count[1]} IDs are undefined)')
    if only_annotate == False:
        mt = mt.filter_cols(mt.eur == 1)
        post_filter_count = mt.count()
        print(f'[get_genetically_european]:{post_filter_count[1]}/{pre_filter_count[1]} IDs were included as genetically european.')
    return mt

In [8]:
result.vep.worst_consequence_by

AttributeError: StructExpression instance has no field, method, or property 'worst_consequence_by'
    Did you mean:
        Data fields: 'worst_consequence_term', 'transcript_consequences', 'most_severe_consequence', 'worst_csq_by_gene', 'motif_feature_consequences'

In [15]:
result.vep.worst_csq_by_gene_canonical.describe()

--------------------------------------------------------
Type:
        array<struct {
        allele_num: int32, 
        amino_acids: str, 
        biotype: str, 
        canonical: int32, 
        ccds: str, 
        cdna_start: int32, 
        cdna_end: int32, 
        cds_end: int32, 
        cds_start: int32, 
        codons: str, 
        consequence_terms: array<str>, 
        distance: int32, 
        domains: array<struct {
            db: str, 
            name: str
        }>, 
        exon: str, 
        gene_id: str, 
        gene_pheno: int32, 
        gene_symbol: str, 
        gene_symbol_source: str, 
        hgnc_id: str, 
        hgvsc: str, 
        hgvsp: str, 
        hgvs_offset: int32, 
        impact: str, 
        intron: str, 
        lof: str, 
        lof_flags: str, 
        lof_filter: str, 
        lof_info: str, 
        minimised: int32, 
        polyphen_prediction: str, 
        polyphen_score: float64, 
        protein_end: int32, 
        protein_s

In [ ]:
#mt.describe()

In [ ]:

#mt.annotate_rows(dbNSFP = )

In [ ]:
#mt.vep.worst_csq_by_gene.gene_symbol.show()

In [15]:
def annotate_dbnsfp(mt, vep_path):
    r'''Annotate matrix table with dbNSFP consequence from external VEP file.'''
    print(f'Annotating with VEP file: {vep_path}')
    
    # Open file containing VEP fields
    with open('data/vep/vep_fields.txt', 'r') as file:
        fields = file.read().strip().split(',')
    ht = hl.import_vcf(vep_path).rename({'info':'vep'}) 
    
    # Add VEP fields by iteration
    for i in range(len(fields)):
        ht = ht.annotate_rows(
            vep=ht.vep.annotate(
                col=ht.vep.CSQ.map(lambda x: (x.split('\\|')[i]))[0]
                ).rename({'col':f'{fields[i]}'})
        )
    
    # Extract various categories annotations and change type
    ht = ht.annotate_rows(vep = ht.vep.annotate(sift_pred = ht.vep.SIFT_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(polyphen2_hdiv_pred = ht.vep.Polyphen2_HDIV_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(polyphen2_hvar_pred = ht.vep.Polyphen2_HVAR_pred.split('&')[0]))
    ht = ht.annotate_rows(vep = ht.vep.annotate(cadd_phred_score = hl.parse_float(ht.vep.CADD_phred)))
    ht = ht.annotate_rows(vep = ht.vep.annotate(revel_score = hl.parse_float(ht.vep.REVEL_score)))
    
    # annotate main table
    mt = mt.annotate_rows(dbnsfp = hl.struct())
    mt = mt.annotate_rows(dbnsfp = mt.dbnsfp.annotate(revel_score = ht.index_rows(mt.locus, mt.alleles).vep.revel_score))
    mt = mt.annotate_rows(dbnsfp = mt.dbnsfp.annotate(cadd_phred_score = ht.index_rows(mt.locus, mt.alleles).vep.cadd_phred_score))
    mt = mt.annotate_rows(dbnsfp = mt.dbnsfp.annotate(polyphen2_hdiv_pred = ht.index_rows(mt.locus, mt.alleles).vep.polyphen2_hdiv_pred))
    mt = mt.annotate_rows(dbnsfp = mt.dbnsfp.annotate(polyphen2_hvar_pred = ht.index_rows(mt.locus, mt.alleles).vep.polyphen2_hvar_pred))

    return(mt)

ht = annotate_dbnsfp(mt, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
ht.vep.consequence_category

Annotating with VEP file: data/vep/full/ukb_wes_200k_full_vep_chr22.vcf


AttributeError: StructExpression instance has no field, method, or property 'consequence_category'
    Did you mean:
        Data field: 'worst_consequence_term'

In [40]:
import re
import sys

def extract_phenos_from_header(path, regexes = ['ID','age','PC+','ukbb', 'sex','array'], delim = '\t'):
    combined = "(" + ")|(".join(regexes) + ")"
    infile = open(path,'r')
    line = infile.readline().strip('\n').split(delim)
    line_clean = [l for l in line if not re.match(combined, l)]
    return line_clean


out = extract_phenos_from_header('/well/lindgren/UKBIOBANK/dpalmer/ukb_wes_phenotypes/200k/UKBB_WES200k_binary_phenotypes.tsv')
sys.stdout.write("\t".join(out))

colorectal_cancer	Trachea_bronchus_lung_cancer	breast_cancer	hypothalamic_amenorrhea	POI	dementia	Alzheimers_disease	depression	autism	ADHD	renal_failure	coronary_artery_disease	ischaemic_heart_disease	stroke_hemorrhagic	stroke	ischaemic_stroke	chronic_obstructive_pulmonary_disease	Crohns_disease	IBD	Cirrhosis	NASH	NAFLD	psoriasis	hyperandrogenism	pcos2b	hematuria	proteinuria	acute_renal_failure	chronic_kidney_disease	male_infertility	oligomenorrhea	habitual_aborter	female_infertility	ectopic_pregnancy	Preeclampsia	GDM	intrahepatic_cholestasis_in_pregnancy	polycystic_kidney_disease	T2D	T1D	GDM2	kallmann_syndrome	E230	genotyping.array2

In [33]:
from ko_utils import phenos

In [35]:
phenos.extract_phenos_from_header()

import sys

def main(args):
    
    out = extract_from_header(args.)
    sys.stdout.write("\t".join(out))

if __name__=='__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--input', default=None, help='Path to phenotype file')
    parser.add_argument('--input_delim', default="\t", help='delimiter for input file')
    args = parser.parse_args()
    main(args)


<function ko_utils.phenos.extract_phenos_from_header(path, regexes=['ID', 'age', 'PC+', 'ukbb', 'sex', 'array'])>

In [20]:
mt = qc.get_table('data/mt/ukb_wes_vep_200_chr22.mt', 'mt')
#mt = annotate_dbnsfp(mt, vep_path = 'data/vep/full/ukb_wes_200k_full_vep_chr22.vcf')
#mt = variant_csqs_builder(mt)
mt = filter_to_european(mt, genetically_european = False)
mt.count()

2021-09-21 17:47:54 Hail: INFO: Reading table without type imputation
  Loading field 'eid' as type str (user-supplied)
  Loading field 'Submitted.Gender' as type str (not specified)
  Loading field 'Inferred.Gender' as type str (not specified)
  Loading field 'in.white.British.ancestry.subset' as type int32 (user-supplied)


[get_genetically_european]: Not all samples IDs mapped perfectly (135/199795 IDs are undefined)
[get_genetically_european]:167052/199795 IDs were included as genetically european.


(302041, 167052)

In [15]:
def filter_to_european(mt, genetically_european = True, only_annotate = False):
    r'''Get white british (app 11867) /well/lindgren/UKBIOBANK/DATA/QC/ukb_sqc_v2.txt
    and genetically european from /well/lindgren/UKBIOBANK/laura/k_means_clustering_pcs/ukbb_genetically_european_k4_4PCs_self_rep_Nov2020.txt''' 
    
    # filter to either genetically european or UKBB 
    if genetically_european:
        ht1 = hl.import_table('/well/lindgren/UKBIOBANK/laura/k_means_clustering_pcs/ukbb_genetically_european_k4_4PCs_self_rep_Nov2020.txt',
            types={'eid': hl.tstr, 'genetically_european': hl.tint32}).key_by('eid')
        mt = mt.annotate_cols(eur = ht1[mt.s].genetically_european)
    else:
        ht2 = hl.import_table('/well/lindgren/flassen/ressources/ukb/white_british/210921_ukbb_white_british_samples.txt',
            types={'eid': hl.tstr, 'in.white.British.ancestry.subset': hl.tint32}).key_by('eid')
        mt = mt.annotate_cols(eur = ht2[mt.s]['in.white.British.ancestry.subset'])
    
    # count and subset
    undefined_eur = mt.aggregate_cols(hl.agg.sum(hl.is_missing(mt.eur)))
    pre_filter_count = mt.count()
    if undefined_eur == pre_filter_count[1]:
        raise ValueError('[get_european]: IDs for genetically europeans does not match keys in MatrixTable!')
    if undefined_eur > 0:
        print(f'[get_european]: Not all samples IDs mapped perfectly ({undefined_eur}/{pre_filter_count[1]} IDs are undefined)')
    if only_annotate == False:
        mt = mt.filter_cols(mt.eur == 1)
        post_filter_count = mt.count()
        print(f'[get_european]:{post_filter_count[1]}/{pre_filter_count[1]} IDs were included as genetically european.')
    return mt

In [9]:
def main(args):

    hail_init.hail_bmrc_init('logs/hail_grm.log', 'GRCh37')
    
    # input variables
    chroms = args.chroms
    subset_markers_by_kinship = args.subset_markers_by_kinship
    out_prefix = args.out_prefix
    subset_samples_by_eur = args.subset_samples_by_eur
        
    # combine multiple matrix tables
    mt = genotypes.get_ukb_imputed_v3_bgen(chroms)
       
    # createse a sparse GRM
    if subset_markers_by_kinship:
        ht = hl.import_table('/well/lindgren/UKBIOBANK/DATA/QC/ukb_snp_qc.txt', impute = True, delimiter = ' ')
        ht = ht.filter(ht.in_Relatedness == 1)
        rsids = ht.rs_id.collect()
        mt = mt.filter_rows(hl.literal(rsids).contains(mt.rsid))
    
    # subset population
    if subset_to_wes200k:
        mt = genotypes.get_ukb_imputed_v3_bgen(chroms)
        ids = get_ukb_wes_200k_post_qc_path(chrom=22).s.collect()
        mt = mt.filter_rows(hl.literal(ids).contains(mt.s))
        
    # get white british ancestry
    if subset_samples_by_eur:
        mt = qc.filter_to_european(mt)
    
    hl.export_plink(mt, out_prefix, ind_id = mt.s)